# GéoMarketing IDF — J7 : Profil détaillé de la clientèle potentielle


## 07a - Population par sexe et âge

## Objectif

Créer le profil démographique de chaque commune francilienne :

- population totale ;
- hommes et femmes ;
- population par tranche d’âge ;
- clientèle potentielle de 15 à 39 ans ;
- jeunes, adultes et seniors ;
- parts en pourcentage.

Ensuite, on produira les indicateurs démographiques communaux
du profil de clientèle à partir du RP2023.

In [12]:
#Importation des librairies
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
from zipfile import ZipFile
import re
import shutil
import unicodedata

import numpy as np
import pandas as pd
import openpyxl

from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)

print("Pandas :", pd.__version__)
print("NumPy :", np.__version__)
print("Openpyxl :", openpyxl.__version__)
print("Importations réussies ✅")

Pandas : 2.2.2
NumPy : 1.26.4
Openpyxl : 3.1.5
Importations réussies ✅


In [13]:
#Dossiers
RACINE = Path(
    r"C:\Users\almou\OneDrive\GeoMarketing_IDF"
)

DOSSIER_RAW = (
    RACINE
    / "data"
    / "raw"
    / "insee"
    / "rp2023"
)

DOSSIER_INTERIM = (
    RACINE
    / "data"
    / "interim"
)

DOSSIER_PROCESSED = (
    RACINE
    / "data"
    / "processed"
)

for dossier in [
    DOSSIER_RAW,
    DOSSIER_INTERIM,
    DOSSIER_PROCESSED,
]:
    dossier.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Racine :", RACINE)
print("Raw :", DOSSIER_RAW)
print("Interim :", DOSSIER_INTERIM)
print("Processed :", DOSSIER_PROCESSED)

assert RACINE.exists(), "Le dossier du projet n'existe pas."

Racine : C:\Users\almou\OneDrive\GeoMarketing_IDF
Raw : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\insee\rp2023
Interim : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim
Processed : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed


In [37]:
#Fonctions communes
def normaliser_nom_colonne(nom):
    nom = str(nom).strip().upper()

    nom = unicodedata.normalize(
        "NFKD",
        nom,
    )

    nom = "".join(
        caractere
        for caractere in nom
        if not unicodedata.combining(caractere)
    )

    nom = re.sub(
        r"[^A-Z0-9]+",
        "_",
        nom,
    )

    return nom.strip("_")


def normaliser_code_commune(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True,
        )
        .str.upper()
        .str.zfill(5)
    )


def pourcentage(numerateur, denominateur):
    numerateur = pd.to_numeric(
        numerateur,
        errors="coerce",
    )

    denominateur = pd.to_numeric(
        denominateur,
        errors="coerce",
    )

    return (
        numerateur
        .div(
            denominateur.where(
                denominateur.ne(0)
            )
        )
        .mul(100)
    )


def verifier_classeur_xlsx(fichier):
    if not fichier.exists():
        raise FileNotFoundError(
            f"Classeur introuvable : {fichier}"
        )

    with open(fichier, "rb") as flux:
        signature = flux.read(4)

    if signature != b"PK\x03\x04":
        raise ValueError(
            f"{fichier.name} n'est pas un véritable fichier XLSX."
        )

    with ZipFile(fichier) as archive:
        noms = set(archive.namelist())

        if "xl/workbook.xml" not in noms:
            raise ValueError(
                f"{fichier.name} ne contient pas de classeur Excel valide."
            )

    print(
        f"Classeur valide : {fichier.name} "
        f"({fichier.stat().st_size / 1_000_000:.2f} Mo)"
    )


def enregistrer_csv(table, fichier):
    table.to_csv(
        fichier,
        index=False,
        sep=",",
        encoding="utf-8-sig",
    )

    print(
        "Fichier CSV créé :",
        fichier,
    )

In [15]:
noms_profils_j6 = [
    "profil_communes_idf_j6.xlsx",
    "profil_communes_idf_j6.csv",
    "profil_communes_idf.csv",
]

FICHIER_PROFIL_J6 = None

for nom in noms_profils_j6:
    candidat = DOSSIER_PROCESSED / nom

    if candidat.exists():
        FICHIER_PROFIL_J6 = candidat
        break

if FICHIER_PROFIL_J6 is None:
    raise FileNotFoundError(
        "Le profil J6 est introuvable dans data/processed."
    )

if FICHIER_PROFIL_J6.suffix.lower() == ".xlsx":
    profil_j6 = pd.read_excel(
        FICHIER_PROFIL_J6,
        sheet_name=0,
        engine="openpyxl",
    )

else:
    profil_j6 = pd.read_csv(
        FICHIER_PROFIL_J6,
        sep=None,
        engine="python",
        encoding="utf-8-sig",
    )

profil_j6.columns = [
    normaliser_nom_colonne(colonne)
    for colonne in profil_j6.columns
]

if "CODGEO" not in profil_j6.columns:
    candidats_code = [
        "DEPCOM",
        "CODE_COMMUNE",
        "COM",
    ]

    colonne_code = next(
        (
            colonne
            for colonne in candidats_code
            if colonne in profil_j6.columns
        ),
        None,
    )

    if colonne_code is None:
        raise ValueError(
            "Aucune colonne de code communal dans le profil J6."
        )

    profil_j6 = profil_j6.rename(
        columns={
            colonne_code: "CODGEO"
        }
    )

profil_j6["CODGEO"] = normaliser_code_commune(
    profil_j6["CODGEO"]
)

assert profil_j6["CODGEO"].notna().all()
assert profil_j6["CODGEO"].is_unique
assert profil_j6["CODGEO"].str.fullmatch(
    r"\d{5}"
).all()

codes_profil = set(
    profil_j6["CODGEO"]
)

print("Profil J6 :", FICHIER_PROFIL_J6)
print("Nombre de communes :", len(profil_j6))
print("Profil J6 chargé ✅")

Profil J6 : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j6.csv
Nombre de communes : 1266
Profil J6 chargé ✅


In [17]:
#Définir le fichier source
FICHIER_POPULATION = (
    DOSSIER_RAW
    / "TD_POP1_2023.xlsx"
)

verifier_classeur_xlsx(
    FICHIER_POPULATION
)

Classeur valide : TD_POP1_2023.xlsx (36.03 Mo)


In [18]:
#Inspecter les onglets
classeur_population = pd.ExcelFile(
    FICHIER_POPULATION,
    engine="openpyxl",
)

print(
    "Onglets :",
    classeur_population.sheet_names,
)

assert "COM" in classeur_population.sheet_names

Onglets : ['Métadonnées', 'COM', 'ARM', 'Documentation']


In [19]:
#Lecture des données
population_source = pd.read_excel(
    FICHIER_POPULATION,
    sheet_name="COM",
    engine="openpyxl",
    dtype={
        "Code géographique": "string",
    },
)

print(
    "Dimensions nationales :",
    population_source.shape,
)

display(
    population_source.iloc[
        :5,
        :10,
    ]
)

Dimensions nationales : (34858, 204)


,Code géographique,Libellé géographique,Homme\n0 an,Homme\n1 an,Homme\n2 ans,Homme\n3 ans,Homme\n4 ans,Homme\n5 ans,Homme\n6 ans,Homme\n7 ans
0,01001,L'Abergement-Clémenciat,6.99962,7.93690,3.98261,2.96757,5.95631,5.95616,15.90921,4.94131
1,01002,L'Abergement-de-Varey,1.96452,0.00000,0.97597,0.00000,1.95193,3.90402,0.00000,1.95193
2,01004,Ambérieu-en-Bugey,108.59939,128.46059,79.12077,151.08195,119.76589,79.59773,108.21020,105.89583
3,01005,Ambérieux-en-Dombes,16.08265,9.79383,9.47741,10.12942,18.32101,7.64442,14.96222,15.27506
4,01006,Ambléon,1.00877,0.00000,1.00877,0.00000,1.00877,0.00000,0.00000,0.00000


In [22]:
#Normaliser les identifiants
population_source = population_source.rename(
    columns={
        "Code géographique": "CODGEO",
        "Libellé géographique": "LIBGEO",
    }
)

population_source["CODGEO"] = (
    normaliser_code_commune(
        population_source["CODGEO"]
    )
)

assert population_source["CODGEO"].is_unique
assert population_source["CODGEO"].notna().all()

print(
    "Nombre de communes nationales :",
    len(population_source),
)

Nombre de communes nationales : 34858


In [23]:
#Reconnaitre les indicateurs "Age" et "Sexe"
def analyser_colonne_age_sexe(colonne):
    texte = re.sub(
        r"\s+",
        " ",
        str(colonne).strip(),
    )

    resultat = re.fullmatch(
        r"(Homme|Femme) "
        r"(\d+) "
        r"ans?"
        r"(?: ou plus)?",
        texte,
        flags=re.IGNORECASE,
    )

    if resultat is None:
        return None

    sexe_texte = resultat.group(1).lower()
    age = int(resultat.group(2))

    sexe = (
        "H"
        if sexe_texte == "homme"
        else "F"
    )

    return sexe, age


informations_colonnes = []

for colonne in population_source.columns:
    resultat = analyser_colonne_age_sexe(
        colonne
    )

    if resultat is not None:
        sexe, age = resultat

        informations_colonnes.append(
            {
                "COLONNE_SOURCE": colonne,
                "SEXE": sexe,
                "AGE": age,
            }
        )

dictionnaire_age = pd.DataFrame(
    informations_colonnes
)

print(
    "Colonnes âge-sexe reconnues :",
    len(dictionnaire_age),
)

display(
    dictionnaire_age.head()
)

assert len(dictionnaire_age) == 202
assert (
    dictionnaire_age["SEXE"]
    .value_counts()
    .to_dict()
    == {
        "H": 101,
        "F": 101,
    }
)

assert dictionnaire_age["AGE"].min() == 0
assert dictionnaire_age["AGE"].max() == 100

Colonnes âge-sexe reconnues : 202


,COLONNE_SOURCE,SEXE,AGE
0,Homme\n0 an,H,0
1,Homme\n1 an,H,1
2,Homme\n2 ans,H,2
3,Homme\n3 ans,H,3
4,Homme\n4 ans,H,4


In [24]:
#Filtrer les communes d'Ile-de-France
codes_population = set(
    population_source["CODGEO"]
)

codes_absents_source = sorted(
    codes_profil - codes_population
)

if codes_absents_source:
    print(
        "Codes du profil absents du fichier population :",
        codes_absents_source,
    )

    raise ValueError(
        "Certaines communes du profil J6 sont absentes du "
        "classeur RP2023. Vérifie notamment la présence "
        "éventuelle de l'ancien code 93059."
    )

population_idf = population_source[
    population_source["CODGEO"].isin(
        codes_profil
    )
].copy()

population_idf = population_idf.sort_values(
    "CODGEO"
).reset_index(drop=True)

print(
    "Communes franciliennes retenues :",
    len(population_idf),
)

assert len(population_idf) == len(profil_j6)

Communes franciliennes retenues : 1266


In [25]:
#Passer au format long
colonnes_age = (
    dictionnaire_age["COLONNE_SOURCE"]
    .tolist()
)

sexe_par_colonne = dict(
    zip(
        dictionnaire_age["COLONNE_SOURCE"],
        dictionnaire_age["SEXE"],
    )
)

age_par_colonne = dict(
    zip(
        dictionnaire_age["COLONNE_SOURCE"],
        dictionnaire_age["AGE"],
    )
)

population_longue = population_idf.melt(
    id_vars=[
        "CODGEO",
        "LIBGEO",
    ],
    value_vars=colonnes_age,
    var_name="COLONNE_SOURCE",
    value_name="EFFECTIF",
)

population_longue["SEXE"] = (
    population_longue["COLONNE_SOURCE"]
    .map(sexe_par_colonne)
)

population_longue["AGE"] = (
    population_longue["COLONNE_SOURCE"]
    .map(age_par_colonne)
    .astype(int)
)

population_longue["EFFECTIF"] = (
    pd.to_numeric(
        population_longue["EFFECTIF"],
        errors="coerce",
    )
)

print(
    "Dimensions de la table longue :",
    population_longue.shape,
)

print(
    "Valeurs numériques absentes :",
    population_longue["EFFECTIF"].isna().sum(),
)

assert population_longue["EFFECTIF"].notna().all()
assert population_longue["EFFECTIF"].ge(0).all()

display(
    population_longue.head()
)

Dimensions de la table longue : (255732, 6)
Valeurs numériques absentes : 0


,CODGEO,LIBGEO,COLONNE_SOURCE,EFFECTIF,SEXE,AGE
0,75056,Paris,Homme\n0 an,9427.90742,H,0
1,77001,Achères-la-Forêt,Homme\n0 an,6.98121,H,0
2,77002,Amillis,Homme\n0 an,2.91855,H,0
3,77003,Amponville,Homme\n0 an,1.93522,H,0
4,77004,Andrezel,Homme\n0 an,4.22624,H,0


In [27]:
#Créer les tranches d'âge
bornes_age = [
    -1,
    2,
    5,
    10,
    14,
    17,
    24,
    39,
    54,
    64,
    79,
    np.inf,
]

noms_tranches_age = [
    "0_2",
    "3_5",
    "6_10",
    "11_14",
    "15_17",
    "18_24",
    "25_39",
    "40_54",
    "55_64",
    "65_79",
    "80_PLUS",
]

population_longue["TRANCHE_AGE"] = pd.cut(
    population_longue["AGE"],
    bins=bornes_age,
    labels=noms_tranches_age,
    right=True,
)

assert population_longue[
    "TRANCHE_AGE"
].notna().all()

display(
    population_longue[
        [
            "AGE",
            "TRANCHE_AGE",
        ]
    ]
    .drop_duplicates()
    .sort_values("AGE")
    .head(30)
)

,AGE,TRANCHE_AGE
0,0,0_2
1266,1,0_2
2532,2,0_2
3798,3,3_5
5064,4,3_5
6330,5,3_5
7596,6,6_10
8862,7,6_10
10128,8,6_10
11394,9,6_10


In [29]:
#Agréger par tranche d'âge
table_age = (
    population_longue
    .groupby(
        [
            "CODGEO",
            "TRANCHE_AGE",
        ],
        observed=False,
    )["EFFECTIF"]
    .sum()
    .unstack(fill_value=0)
)

table_age.columns = [
    f"POP_{tranche}"
    for tranche in table_age.columns
]

table_age = table_age.reset_index()

display(
    table_age.head()
)

,CODGEO,POP_0_2,POP_3_5,POP_6_10,POP_11_14,POP_15_17,POP_18_24,POP_25_39,POP_40_54,POP_55_64,POP_65_79,POP_80_PLUS
0,75056,54155.23178,51427.58154,87388.40827,75287.99165,59522.89891,236306.98805,540939.72777,390054.10537,236685.88601,263552.66604,108456.51460
1,77001,32.18226,25.01503,65.89736,58.31802,48.54337,66.60957,156.30366,299.47969,205.82706,168.24409,64.57987
2,77002,28.18949,28.14560,43.59675,31.94438,30.93410,55.54298,135.20762,153.20499,135.60447,132.30342,55.32622
3,77003,9.79907,10.83316,23.50828,17.64146,4.93543,17.77637,59.55102,79.40904,56.23723,55.18079,14.12806
4,77004,22.08237,18.84292,24.37632,15.93868,7.29929,12.49477,71.70242,59.20604,43.99085,52.96427,7.10208


In [30]:
#Agréger par sexe
table_sexe = (
    population_longue
    .groupby(
        [
            "CODGEO",
            "SEXE",
        ]
    )["EFFECTIF"]
    .sum()
    .unstack(fill_value=0)
    .rename(
        columns={
            "H": "POP_HOMMES",
            "F": "POP_FEMMES",
        }
    )
    .reset_index()
)

display(
    table_sexe.head()
)

SEXE,CODGEO,POP_FEMMES,POP_HOMMES
0,75056,1.115440e+06,988338.37118
1,77001,6.173427e+02,573.65727
2,77002,4.261917e+02,403.80833
3,77003,1.780602e+02,170.93972
4,77004,1.699077e+02,166.09235


In [31]:
#Créer les croisements Sexe et Age
table_sexe_age = (
    population_longue
    .groupby(
        [
            "CODGEO",
            "SEXE",
            "TRANCHE_AGE",
        ],
        observed=False,
    )["EFFECTIF"]
    .sum()
    .unstack(
        [
            "SEXE",
            "TRANCHE_AGE",
        ],
        fill_value=0,
    )
)

table_sexe_age.columns = [
    f"POP_{sexe}_{tranche}"
    for sexe, tranche
    in table_sexe_age.columns
]

table_sexe_age = (
    table_sexe_age
    .reset_index()
)

display(
    table_sexe_age.head()
)

,CODGEO,POP_F_0_2,POP_F_3_5,POP_F_6_10,POP_F_11_14,POP_F_15_17,POP_F_18_24,POP_F_25_39,POP_F_40_54,POP_F_55_64,POP_F_65_79,POP_F_80_PLUS,POP_H_0_2,POP_H_3_5,POP_H_6_10,POP_H_11_14,POP_H_15_17,POP_H_18_24,POP_H_25_39,POP_H_40_54,POP_H_55_64,POP_H_65_79,POP_H_80_PLUS
0,75056,26522.40142,25422.98841,42844.68737,37530.31927,29670.16369,132291.74878,279018.79281,199372.77054,124520.14053,148783.29856,69462.31743,27632.83036,26004.59313,44543.72090,37757.67238,29852.73522,104015.23927,261920.93496,190681.33483,112165.74548,114769.36748,38994.19717
1,77001,17.12594,11.98989,34.99483,28.04483,26.32065,32.30150,84.17936,158.49482,98.48529,84.34948,41.05612,15.05632,13.02514,30.90253,30.27319,22.22272,34.30807,72.12430,140.98487,107.34177,83.89461,23.52375
2,77002,15.51577,16.43474,17.31972,15.62663,20.24866,23.28392,67.94481,72.47546,68.93133,64.23183,44.17882,12.67372,11.71086,26.27703,16.31775,10.68544,32.25906,67.26281,80.72953,66.67314,68.07159,11.14740
3,77003,2.92853,5.88732,8.83634,9.80862,3.95253,10.87492,33.71296,38.77357,27.09351,28.09745,8.09444,6.87054,4.94584,14.67194,7.83284,0.98290,6.90145,25.83806,40.63547,29.14372,27.08334,6.03362
4,77004,10.54011,9.45491,10.55871,7.46867,5.22811,5.21166,40.54601,25.94448,19.36491,29.50715,6.08294,11.54226,9.38801,13.81761,8.47001,2.07118,7.28311,31.15641,33.26156,24.62594,23.45712,1.01914


In [32]:
#Construire les indicateurs
indicateurs_7a = (
    population_idf[
        [
            "CODGEO",
            "LIBGEO",
        ]
    ]
    .merge(
        table_age,
        on="CODGEO",
        how="left",
        validate="one_to_one",
    )
    .merge(
        table_sexe,
        on="CODGEO",
        how="left",
        validate="one_to_one",
    )
    .merge(
        table_sexe_age,
        on="CODGEO",
        how="left",
        validate="one_to_one",
    )
)

indicateurs_7a["POP_RP2023"] = (
    indicateurs_7a["POP_HOMMES"]
    + indicateurs_7a["POP_FEMMES"]
)

indicateurs_7a["POP_0_17"] = (
    indicateurs_7a["POP_0_2"]
    + indicateurs_7a["POP_3_5"]
    + indicateurs_7a["POP_6_10"]
    + indicateurs_7a["POP_11_14"]
    + indicateurs_7a["POP_15_17"]
)

indicateurs_7a["POP_15_24"] = (
    indicateurs_7a["POP_15_17"]
    + indicateurs_7a["POP_18_24"]
)

indicateurs_7a["POP_15_39"] = (
    indicateurs_7a["POP_15_17"]
    + indicateurs_7a["POP_18_24"]
    + indicateurs_7a["POP_25_39"]
)

indicateurs_7a["POP_18_39"] = (
    indicateurs_7a["POP_18_24"]
    + indicateurs_7a["POP_25_39"]
)

indicateurs_7a["POP_65_PLUS"] = (
    indicateurs_7a["POP_65_79"]
    + indicateurs_7a["POP_80_PLUS"]
)

indicateurs_7a["PART_FEMMES_PCT"] = (
    pourcentage(
        indicateurs_7a["POP_FEMMES"],
        indicateurs_7a["POP_RP2023"],
    )
)

indicateurs_7a["PART_0_17_PCT"] = (
    pourcentage(
        indicateurs_7a["POP_0_17"],
        indicateurs_7a["POP_RP2023"],
    )
)

indicateurs_7a["PART_15_24_PCT"] = (
    pourcentage(
        indicateurs_7a["POP_15_24"],
        indicateurs_7a["POP_RP2023"],
    )
)

indicateurs_7a["PART_15_39_PCT"] = (
    pourcentage(
        indicateurs_7a["POP_15_39"],
        indicateurs_7a["POP_RP2023"],
    )
)

indicateurs_7a["PART_18_39_PCT"] = (
    pourcentage(
        indicateurs_7a["POP_18_39"],
        indicateurs_7a["POP_RP2023"],
    )
)

indicateurs_7a["PART_65_PLUS_PCT"] = (
    pourcentage(
        indicateurs_7a["POP_65_PLUS"],
        indicateurs_7a["POP_RP2023"],
    )
)

indicateurs_7a = indicateurs_7a.sort_values(
    "CODGEO"
).reset_index(drop=True)

display(
    indicateurs_7a.head()
)

,CODGEO,LIBGEO,POP_0_2,POP_3_5,POP_6_10,POP_11_14,POP_15_17,POP_18_24,POP_25_39,POP_40_54,POP_55_64,POP_65_79,POP_80_PLUS,POP_FEMMES,POP_HOMMES,POP_F_0_2,POP_F_3_5,POP_F_6_10,POP_F_11_14,POP_F_15_17,POP_F_18_24,POP_F_25_39,POP_F_40_54,POP_F_55_64,POP_F_65_79,POP_F_80_PLUS,POP_H_0_2,POP_H_3_5,POP_H_6_10,POP_H_11_14,POP_H_15_17,POP_H_18_24,POP_H_25_39,POP_H_40_54,POP_H_55_64,POP_H_65_79,POP_H_80_PLUS,POP_RP2023,POP_0_17,POP_15_24,POP_15_39,POP_18_39,POP_65_PLUS,PART_FEMMES_PCT,PART_0_17_PCT,PART_15_24_PCT,PART_15_39_PCT,PART_18_39_PCT,PART_65_PLUS_PCT
0,75056,Paris,54155.23178,51427.58154,87388.40827,75287.99165,59522.89891,236306.98805,540939.72777,390054.10537,236685.88601,263552.66604,108456.51460,1.115440e+06,988338.37118,26522.40142,25422.98841,42844.68737,37530.31927,29670.16369,132291.74878,279018.79281,199372.77054,124520.14053,148783.29856,69462.31743,27632.83036,26004.59313,44543.72090,37757.67238,29852.73522,104015.23927,261920.93496,190681.33483,112165.74548,114769.36748,38994.19717,2.103778e+06,327782.11215,295829.88696,836769.61473,777246.71582,372009.18064,53.020786,15.580642,14.061840,39.774616,36.945282,17.682910
1,77001,Achères-la-Forêt,32.18226,25.01503,65.89736,58.31802,48.54337,66.60957,156.30366,299.47969,205.82706,168.24409,64.57987,6.173427e+02,573.65727,17.12594,11.98989,34.99483,28.04483,26.32065,32.30150,84.17936,158.49482,98.48529,84.34948,41.05612,15.05632,13.02514,30.90253,30.27319,22.22272,34.30807,72.12430,140.98487,107.34177,83.89461,23.52375,1.191000e+03,229.95604,115.15294,271.45660,222.91323,232.82396,51.833982,19.307812,9.668593,22.792326,18.716476,19.548612
2,77002,Amillis,28.18949,28.14560,43.59675,31.94438,30.93410,55.54298,135.20762,153.20499,135.60447,132.30342,55.32622,4.261917e+02,403.80833,15.51577,16.43474,17.31972,15.62663,20.24866,23.28392,67.94481,72.47546,68.93133,64.23183,44.17882,12.67372,11.71086,26.27703,16.31775,10.68544,32.25906,67.26281,80.72953,66.67314,68.07159,11.14740,8.300000e+02,162.81032,86.47708,221.68470,190.75060,187.62964,51.348395,19.615701,10.418925,26.708999,22.981999,22.605980
3,77003,Amponville,9.79907,10.83316,23.50828,17.64146,4.93543,17.77637,59.55102,79.40904,56.23723,55.18079,14.12806,1.780602e+02,170.93972,2.92853,5.88732,8.83634,9.80862,3.95253,10.87492,33.71296,38.77357,27.09351,28.09745,8.09444,6.87054,4.94584,14.67194,7.83284,0.98290,6.90145,25.83806,40.63547,29.14372,27.08334,6.03362,3.489999e+02,66.71740,22.71180,82.26282,77.32739,69.30885,51.020125,19.116738,6.507681,23.571015,22.156851,19.859274
4,77004,Andrezel,22.08237,18.84292,24.37632,15.93868,7.29929,12.49477,71.70242,59.20604,43.99085,52.96427,7.10208,1.699077e+02,166.09235,10.54011,9.45491,10.55871,7.46867,5.22811,5.21166,40.54601,25.94448,19.36491,29.50715,6.08294,11.54226,9.38801,13.81761,8.47001,2.07118,7.28311,31.15641,33.26156,24.62594,23.45712,1.01914,3.360000e+02,88.53958,19.79406,91.49648,84.19719,60.06635,50.567754,26.351065,5.891089,27.231094,25.058687,17.876889


In [33]:
colonnes_tranches = [
    f"POP_{tranche}"
    for tranche in noms_tranches_age
]

somme_tranches = (
    indicateurs_7a[colonnes_tranches]
    .sum(axis=1)
)

ecart_age_total = (
    somme_tranches
    - indicateurs_7a["POP_RP2023"]
).abs()

ecart_sexe_total = (
    indicateurs_7a["POP_HOMMES"]
    + indicateurs_7a["POP_FEMMES"]
    - indicateurs_7a["POP_RP2023"]
).abs()

assert ecart_age_total.max() < 0.01
assert ecart_sexe_total.max() < 0.01
assert indicateurs_7a["CODGEO"].is_unique
assert len(indicateurs_7a) == len(profil_j6)
assert "75056" in indicateurs_7a["CODGEO"].values
assert "93066" in indicateurs_7a["CODGEO"].values

if "93059" in indicateurs_7a["CODGEO"].values:
    raise ValueError(
        "L'ancien code de Pierrefitte-sur-Seine 93059 "
        "est encore présent."
    )

controle_7a = pd.DataFrame(
    {
        "CONTROLE": [
            "Nombre de communes",
            "Nombre de colonnes âge-sexe",
            "Écart maximal somme tranches",
            "Écart maximal somme sexes",
            "Présence de Paris 75056",
            "Présence de Saint-Denis 93066",
        ],
        "VALEUR": [
            len(indicateurs_7a),
            len(dictionnaire_age),
            ecart_age_total.max(),
            ecart_sexe_total.max(),
            "75056" in indicateurs_7a["CODGEO"].values,
            "93066" in indicateurs_7a["CODGEO"].values,
        ],
    }
)

display(controle_7a)

print("Contrôles J7a validés ✅")

,CONTROLE,VALEUR
0,Nombre de communes,1266
1,Nombre de colonnes âge-sexe,202
2,Écart maximal somme tranches,0.0
3,Écart maximal somme sexes,0.0
4,Présence de Paris 75056,True
5,Présence de Saint-Denis 93066,True


Contrôles J7a validés ✅


In [38]:
FICHIER_SORTIE_7A = (
    DOSSIER_INTERIM
    / "j7a_population_age_sexe_idf_2023.csv"
)

FICHIER_CONTROLE_7A = (
    DOSSIER_INTERIM
    / "j7a_controle_population_age_sexe.csv"
)

FICHIER_DICTIONNAIRE_7A = (
    DOSSIER_INTERIM
    / "j7a_dictionnaire_age_sexe.csv"
)

enregistrer_csv(
    indicateurs_7a,
    FICHIER_SORTIE_7A,
)

enregistrer_csv(
    controle_7a,
    FICHIER_CONTROLE_7A,
)

enregistrer_csv(
    dictionnaire_age,
    FICHIER_DICTIONNAIRE_7A,
)

print("J7a terminé ✅")

Fichier CSV créé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j7a_population_age_sexe_idf_2023.csv
Fichier CSV créé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j7a_controle_population_age_sexe.csv
Fichier CSV créé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j7a_dictionnaire_age_sexe.csv
J7a terminé ✅
